In [ ]:
import os
import sys
from pathlib import Path

# Set these before importing any `reva` modules. Update the placeholder paths
# to match your machine or shared Jupyter environment.
os.environ["REVA_DATA_ROOT"] = "/path/to/your/data/root"
os.environ["REVA_HF_CACHE_ROOT"] = "/path/to/your/hf_cache/root"
os.environ["REVA_CHECKPOINT_ROOT"] = "/path/to/your/checkpoints/root"
os.environ["REVA_EVAL_RESULTS_ROOT"] = "/path/to/your/eval_results/root"
os.environ["REVA_REGION_DATA_ROOT"] = "/path/to/your/region_data/root"
os.environ["REVA_DECONTAMINATION_ROOT"] = "/path/to/your/decontamination/root"
os.environ["REVA_GROUNDING_DINO_ROOT"] = "/path/to/your/groundingdino/root"
os.environ["REVA_VQAV2_ROOT"] = "/path/to/your/vqav2/root"
os.environ["REVA_TEST_IMAGES_ROOT"] = "/path/to/your/test_images/root"

hf_cache_root = Path(os.environ["REVA_HF_CACHE_ROOT"]).expanduser()
os.environ["HF_HOME"] = str(hf_cache_root)
os.environ["HF_HUB_CACHE"] = str(hf_cache_root / "hub")
os.environ["HF_DATASETS_CACHE"] = str(hf_cache_root / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache_root / "hub")

for var_name in (
    "REVA_DATA_ROOT",
    "REVA_HF_CACHE_ROOT",
    "REVA_CHECKPOINT_ROOT",
    "REVA_EVAL_RESULTS_ROOT",
    "REVA_REGION_DATA_ROOT",
    "REVA_DECONTAMINATION_ROOT",
    "REVA_GROUNDING_DINO_ROOT",
    "REVA_VQAV2_ROOT",
    "REVA_TEST_IMAGES_ROOT",
    "HF_HOME",
    "HF_HUB_CACHE",
    "HF_DATASETS_CACHE",
    "TRANSFORMERS_CACHE",
):
    print(f"{var_name} = {os.environ.get(var_name)}")


def find_reva_project_root(start: Path) -> Path:
    override = os.environ.get("REVA_PROJECT_DIR")
    if override:
        return Path(override).expanduser().resolve()

    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "reva" / "evaluation.py").is_file() and (candidate / "reva" / "config.py").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate the ReVA project root from the current working directory. "
        "Set REVA_PROJECT_DIR to your cloned repo path."
    )


PROJECT_ROOT = find_reva_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())


## Install dependencies

In [ ]:
!pip install transformers==4.45.0 datasets accelerate pillow bitsandbytes sentencepiece tqdm matplotlib "numpy<2.0" --quiet

In [ ]:
from pprint import pprint
import zipfile
from huggingface_hub import hf_hub_download

# Load only the conversations JSON file, not the full repo.
# Because loading 'liuhaotian/LLaVA-Pretrain' directly causes a CastError because the repo contains two JSON files with different schemas. 
# Pointing at the specific file that has the 'conversations' field avoids this entirely.
raw_dataset = load_dataset("json", data_files="hf://datasets/liuhaotian/LLaVA-Pretrain/blip_laion_cc_sbu_558k.json", split="train")
snapshot_dir = "~/reva-data/hf_cache/hub/datasets--liuhaotian--LLaVA-Pretrain/snapshots/70f9d1e5e1a697fe35830875cfc7de1dd590d727"

zip_path = hf_hub_download(
    repo_id="liuhaotian/LLaVA-Pretrain",
    filename="images.zip",
    repo_type="dataset",
    cache_dir="~/reva-data/hf_cache",
)
print(f"Downloaded to: {zip_path}")

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(snapshot_dir)
print("Done. Contents:", os.listdir(snapshot_dir)[:5])
print(f"\nDataset loaded: {len(raw_dataset):,} samples")
#print(os.listdir(snapshot_dir))
#example = raw_dataset[0]
#pprint(example)

## Data decontamination on LLaVA 558k (LAION-CC-SBU (LCS) 558K) removing COCO & VG image overlaps

In [ ]:
!pip install faiss-gpu-cu12 --quiet

In [ ]:
!pip install ImageHash --quiet

In [ ]:
# LCS 558K visual decontamination
# Stage 1: pHash (Hamming <= 4) for pixel-identical / minimally compressed copies
# Stage 2: SSCD (threshold=0.75) for re-encoded, resized, compressed copies

import urllib.request
import zipfile
import numpy as np
import torch
import faiss
import imagehash
import torchvision.transforms as T
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from concurrent.futures import ThreadPoolExecutor, as_completed

PHASH_SIZE = 16
HAMMING_THRESH = 4
HAMMING_BATCH = 256
SSCD_THRESH = 0.75
EMBED_BATCH = 64
NUM_WORKERS = 4

print("Stage 0: collecting reference image pools")

COCO_SPLITS = {
    "train2014": "http://images.cocodataset.org/zips/train2014.zip",
    "val2014": "http://images.cocodataset.org/zips/val2014.zip",
    # test2015 already downloaded
}

for split, url in COCO_SPLITS.items():
    split_dir = vqav2_dir / split
    if split_dir.exists():
        n = sum(1 for _ in split_dir.glob("*.jpg"))
        print(f"COCO {split} already present: {n:,} images")
        continue
    print(f"downloading COCO {split} ...")
    zip_path = vqav2_dir / f"{split}.zip"
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(vqav2_dir)
    print(f"COCO {split} ready.")

coco_image_paths = (
    list((vqav2_dir / "train2014").glob("*.jpg")) +
    list((vqav2_dir / "val2014").glob("*.jpg")) +
    list((vqav2_dir / "test2015").glob("*.jpg"))
)
vg_image_paths = list(gqa_images_dir.glob("*.jpg"))
all_ref_paths = coco_image_paths + vg_image_paths

# sanity check: verify test2015 is present
n_test2015 = sum(1 for _ in (vqav2_dir / "test2015").glob("*.jpg"))
print(f"COCO test2015 present: {n_test2015:,} images")
print(f"COCO total: {len(coco_image_paths):,}, VG: {len(vg_image_paths):,}, total ref: {len(all_ref_paths):,}")
assert len(coco_image_paths) > 200000, f"expected >200K COCO images, got {len(coco_image_paths):,} - check test2015"

print("Stage 1: pHash Hamming <=", HAMMING_THRESH)

popcount_lut = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def compute_phash_bits(path):
    try:
        h = imagehash.phash(Image.open(path).convert("RGB"), hash_size=PHASH_SIZE)
        return h.hash.flatten()   # 256 bool array
    except Exception:
        return None

def build_hash_matrix(paths, desc):
    # submit with explicit dataset index so as_completed order does not matter
    results = {}
    with ThreadPoolExecutor(max_workers=16) as ex:
        futures = {ex.submit(compute_phash_bits, p): idx for idx, p in enumerate(paths)}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=desc):
            idx = futures[fut]
            v = fut.result()
            if v is not None:
                results[idx] = v
    # sort by original index to guarantee alignment between matrix rows and indices
    sorted_items = sorted(results.items())
    valid_indices = [idx for idx, _ in sorted_items]
    matrix = np.packbits(np.array([v for _, v in sorted_items], dtype=bool), axis=1)
    return matrix, valid_indices   # matrix row i corresponds to valid_indices[i]

ref_packed, _ = build_hash_matrix(all_ref_paths, "hashing reference")

train_paths = [Path(images_root_dir) / raw_dataset[i]["image"] for i in range(len(raw_dataset))]
train_packed, train_valid_indices = build_hash_matrix(train_paths, "hashing training")

corrupt_indices = set(range(len(raw_dataset))) - set(train_valid_indices)
print(f"corrupt/unreadable: {len(corrupt_indices):,}")

phash_removed = []
for i in tqdm(range(0, len(train_packed), HAMMING_BATCH), desc="hamming scan"):
    chunk = train_packed[i:i + HAMMING_BATCH]
    xor = np.bitwise_xor(chunk[:, None, :], ref_packed[None, :, :])
    # popcount_lut maps each byte to its number of set bits
    # sum across axis=2 (32 packed bytes) gives total differing bits across 256-bit hash
    hamming = popcount_lut[xor].sum(axis=2)
    matches = np.any(hamming <= HAMMING_THRESH, axis=1)
    for local_idx, is_match in enumerate(matches):
        if is_match:
            # train_valid_indices[i + local_idx] is guaranteed correct
            # because build_hash_matrix sorts by original index
            phash_removed.append(train_valid_indices[i + local_idx])

phash_removed_set = set(phash_removed)
print(f"pHash removed: {len(phash_removed_set):,}")

print("Stage 2: SSCD threshold =", SSCD_THRESH)

sscd_path = Path("~/reva-data/sscd_disc_mixup.torchscript.pt")
if not sscd_path.exists():
    print("downloading SSCD model ...")
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/sscd-copy-detection/sscd_disc_mixup.torchscript.pt", sscd_path)
sscd = torch.jit.load(str(sscd_path)).to(config.device)
sscd.eval()
print("SSCD model loaded.")

sscd_transform = T.Compose([
    T.Resize(288),
    T.CenterCrop(288),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class SSCDDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]).convert("RGB")
            return self.transform(img), True
        except Exception:
            return torch.zeros(3, 288, 288), False

@torch.no_grad()
def embed_sscd(paths, desc):
    dataset = SSCDDataset(paths, sscd_transform)
    loader = DataLoader(dataset, batch_size=EMBED_BATCH, num_workers=NUM_WORKERS, pin_memory=True)
    parts = []
    flags = []
    for batch, batch_flags in tqdm(loader, desc=desc):
        batch = batch.to(config.device)
        feats = sscd(batch).float()
        # explicit L2 normalisation so IndexFlatIP == cosine similarity
        feats = feats / feats.norm(dim=-1, keepdim=True)
        parts.append(feats.cpu().numpy())
        flags.extend(batch_flags.numpy().tolist())
    return np.concatenate(parts, axis=0), np.array(flags, dtype=bool)

print("embedding reference images ...")
ref_embeds, ref_flags = embed_sscd(all_ref_paths, "embedding reference")
ref_embeds = ref_embeds[ref_flags]
# normalise again after slicing to guard against any floating point drift
ref_embeds = ref_embeds / np.linalg.norm(ref_embeds, axis=1, keepdims=True)

DIM = ref_embeds.shape[1]
index = faiss.IndexFlatIP(DIM)
if torch.cuda.is_available():
    res = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, index)
index.add(ref_embeds.astype(np.float32))
print(f"FAISS index: {index.ntotal:,} vectors (dim={DIM})")

sscd_scan_indices = [i for i in range(len(raw_dataset)) if i not in phash_removed_set and i not in corrupt_indices]
sscd_scan_paths = [Path(images_root_dir) / raw_dataset[i]["image"] for i in sscd_scan_indices]

print(f"embedding {len(sscd_scan_paths):,} remaining training images ...")
train_embeds, train_flags = embed_sscd(sscd_scan_paths, "embedding training")

for local_idx, is_valid in enumerate(train_flags):
    if not is_valid:
        corrupt_indices.add(sscd_scan_indices[local_idx])

valid_embeds = train_embeds[train_flags]
valid_embeds = valid_embeds / np.linalg.norm(valid_embeds, axis=1, keepdims=True)
valid_indices = [sscd_scan_indices[i] for i, f in enumerate(train_flags) if f]

D, _ = index.search(valid_embeds.astype(np.float32), k=1)
top_sims = D[:, 0]

print("SSCD similarity distribution:")
for t in [0.95, 0.90, 0.85, 0.80, 0.75, 0.70]:
    print(f"  sim >= {t:.2f}: {int((top_sims >= t).sum()):,}")

sscd_removed = [valid_indices[i] for i, sim in enumerate(top_sims) if sim >= SSCD_THRESH]
print(f"SSCD removed: {len(sscd_removed):,}")

print("Stage 3: applying filter")

all_removed = phash_removed_set | set(sscd_removed) | corrupt_indices
final_clean = sorted(set(range(len(raw_dataset))) - all_removed)

n_total = len(raw_dataset)
print(f"original: {n_total:,}")
print(f"pHash removed: {len(phash_removed_set):,} ({len(phash_removed_set) / n_total * 100:.3f}%)")
print(f"SSCD removed: {len(sscd_removed):,} ({len(sscd_removed) / n_total * 100:.3f}%)")
print(f"corrupt: {len(corrupt_indices):,} ({len(corrupt_indices) / n_total * 100:.3f}%)")
print(f"final clean: {len(final_clean):,} ({len(final_clean) / n_total * 100:.3f}%)")

raw_dataset = raw_dataset.select(final_clean)
print(f"raw_dataset decontaminated: {len(raw_dataset):,} samples")
print("proceed to LLaVAPretrainDataset construction as normal.")

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

phash_set = phash_removed_set
sscd_set = set(sscd_removed)
all_flagged = phash_set | sscd_set

phash_only = sorted(phash_set - sscd_set)
sscd_only = sorted(sscd_set - phash_set)
both = sorted(phash_set & sscd_set)

print(f"total flagged: {len(all_flagged)}")
print(f"pHash only: {len(phash_only)}")
print(f"SSCD only: {len(sscd_only)}")
print(f"both: {len(both)}")

rows = []

sscd_display = sorted(sscd_set)
sscd_local = [valid_indices.index(i) for i in sscd_display if i in valid_indices]
if sscd_local:
    D_sscd, I_sscd = index.search(valid_embeds[sscd_local].astype(np.float32), k=1)
    for k, global_idx in enumerate(sscd_display):
        ref_path = all_ref_paths[I_sscd[k, 0]]
        sim = D_sscd[k, 0]
        method = "pHash+SSCD" if global_idx in phash_set else "SSCD"
        rows.append((global_idx, ref_path, sim, method))

for global_idx in phash_only:
    train_path = Path(images_root_dir) / raw_dataset[global_idx]["image"]
    train_h = compute_phash_bits(train_path)
    best_ref, best_dist = None, 999
    if train_h is not None:
        train_packed_single = np.packbits(train_h.reshape(1, -1), axis=1)
        xor = np.bitwise_xor(train_packed_single[:, None, :], ref_packed[None, :, :])
        dists = popcount_lut[xor].sum(axis=2)[0]
        best_idx = int(dists.argmin())
        best_ref = all_ref_paths[best_idx]
        best_dist = int(dists[best_idx])
    rows.append((global_idx, best_ref, best_dist, "pHash"))

fig, axes = plt.subplots(len(rows), 2, figsize=(8, 4 * len(rows)))
if len(rows) == 1:
    axes = [axes]

for plot_idx, (global_idx, ref_path, sim, method) in enumerate(rows):
    train_path = Path(images_root_dir) / raw_dataset[global_idx]["image"]
    ref_source = "COCO" if ref_path in set(coco_image_paths) else "VG"
    sim_label = f"hamming={sim}" if method == "pHash" else f"sim={sim:.3f}"
    try:
        train_img = Image.open(train_path).convert("RGB")
    except Exception:
        train_img = Image.new("RGB", (224, 224))
    try:
        ref_img = Image.open(ref_path).convert("RGB")
    except Exception:
        ref_img = Image.new("RGB", (224, 224))
    axes[plot_idx][0].imshow(train_img)
    axes[plot_idx][0].set_title(f"training #{global_idx}\n{train_path.name}\nflagged by: {method}", fontsize=8)
    axes[plot_idx][0].axis("off")
    axes[plot_idx][1].imshow(ref_img)
    axes[plot_idx][1].set_title(f"matched {ref_source} ({sim_label})\n{ref_path.name}", fontsize=8)
    axes[plot_idx][1].axis("off")

plt.suptitle(f"flagged training images", fontsize=10)
plt.tight_layout()
plt.show()